### 2. YOLO 모델 결과 분석 및 졸음 감지 로직 구현 🕵️‍♀️

지난 시간에 우리가 직접 학습시킨 **YOLO 모델(`best.pt`)**을 불러와서 테스트해볼 시간입니다.
이번 시간에는 모델이 찾아낸 얼굴의 특징점(Keypoints) 번호를 직접 확인하고, 이를 이용해 졸음을 판단하는 코드를 작성해 봅니다.

### 🚀 미션 목표
1. **모델 로드:** 학습된 YOLO 모델 불러오기
2. **번호 확인(Mapping):** 모델이 예측한 점들 중 '눈'은 몇 번 점인지 시각화하여 찾아내기
3. **알고리즘 구현:** 눈의 특징점을 연결하여 EAR(눈 뜸 정도) 계산하기


### 🎯 [미션] 주석이 달린 줄의 None 또는 _______ 부분을 채워 코드를 완성하세요.

각 셀을 순서대로 실행(Shift + Enter)해야 합니다.

---

#### 1️⃣  필요한 라이브러리 설치 및 불러오기

In [ ]:
# YOLO 라이브러리 설치 (최초 1회만 실행)
!pip install ultralytics

- YOLO 모델을 사용하기 위해서는 `ultralytics` 라이브러리의 핵심 모듈을 불러와야 합니다.

In [ ]:
from ultralytics import YOLO
from ultralytics import settings
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import distance as dist

---
#### 2️⃣ 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cd "/content/drive/MyDrive/2026_AI_Advanced_Study-week4/4차시/02_EAR/code"

In [ ]:
# 현재 작업 디렉토리 확인
current_dir = os.getcwd()
new_runs_dir = os.path.abspath(os.path.join(current_dir, '../runs'))
new_data_dir = os.path.abspath(os.path.join(current_dir, '../data'))
# YOLO 설정 업데이트
settings.update({"runs_dir": new_runs_dir})
settings.update({"datasets_dir": new_data_dir})
settings.update({"wandb": False})  # wandb 로깅 비활성화

print(f"✅ 작업 디렉토리: {current_dir}")

print("\n✅ 경로 설정 완료!")

---
### 3️⃣ 학습된 모델 불러오기 🔍
train.ipynb에서 학습한 모델의 경로를 찾아서 아래 MODEL_PATH 변수에 입력해주세요.

In [ ]:
# 🎯 [미션] (01_train.ipynb에서 학습 후 생성된 weights/best.pt 파일 경로를 지정하세요)
MODEL_PATH =  ______

if os.path.exists(MODEL_PATH):
    model = YOLO(MODEL_PATH)
    print("✅ 모델 로딩 완료!")
else:
    print(f"❌ 모델 파일을 찾을 수 없습니다: {MODEL_PATH}")

---
---
### 4️⃣ 내 모델은 눈을 몇 번 점으로 생각할까? 🤔

우리가 학습시킨 모델이 얼굴의 점을 0번부터 순서대로 찾습니다.

하지만 몇 번 점이 '왼쪽 눈꼬리'인지, '오른쪽 눈꺼풀'인지 우리는 아직 모릅니다.

아래 코드를 실행해서 **얼굴 위에 그려진 번호**를 보고, 눈에 해당하는 번호를 직접 찾아보세요!

In [ ]:
def plot_numbered_keypoints(image_path, model):
    # 이미지 예측 실행
    results = model(image_path)
    
    # 원본 이미지 가져오기
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # 예측된 키포인트(점) 좌표 가져오기
    # results[0].keypoints.xy는 (N, 2) 형태의 좌표 배열입니다.
    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    
    plt.figure(figsize=(24, 24))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title("Original Image")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(img)
    # 각 점 위에 '번호(Index)' 쓰기
    for i, point in enumerate(keypoints):
        x, y = point
        if x == 0 and y == 0: continue # 감지되지 않은 점은 패스
        
        # 점 그리기 (빨간색)
        plt.scatter(x, y, s=20, c='red', marker='o')
        # 번호 쓰기 (파란색, 글자 크기 키움)
        plt.text(x+2, y-2, str(i), fontsize=12, color='blue', weight='bold')

    plt.title("Face Keypoints Index Map")
    plt.axis('off')
    plt.show()

# '눈 뜬 사진'을 넣어 번호를 확인해봅시다.
plot_numbered_keypoints('../data/test/images/26638112_1.jpg', model)

---

#### 5️⃣ 눈의 위치(Index) 찾기 🔍👁️

AI 모델은 얼굴에서 수십 개의 점(Keypoints)을 찾습니다. 그중에서 **"눈"**에 해당하는 점이 몇 번인지 알려줘야 합니다.

**오른쪽 눈**을 감싸고 있는 점 6개의 번호와, **왼쪽 눈**을 감싸고 있는 점 6개의 번호를 순서대로 적어주세요.
- 순서 : [눈끝, 눈 상단 바깥, 눈 상단 안쪽, 눈 안쪽, 눈 하단 안쪽, 눈 하단 바깥]

힌트: 보통 68개 랜드마크 기준, 오른쪽 눈은 36~41번, 왼쪽 눈은 42~47번입니다.

In [ ]:
# --- [학생 미션 1] 눈의 좌표 인덱스 리스트 완성하기 ---

# 🎯 [미션] 오른쪽 눈 점 6개 번호 리스트
RIGHT_EYE_IDXS = [ , , , , , ]

# 🎯 [미션] 왼쪽 눈 점 6개 번호 리스트
LEFT_EYE_IDXS = [ , , , , , ]

print(f"설정된 인덱스 - 우안: {RIGHT_EYE_IDXS}, 좌안: {LEFT_EYE_IDXS}")

---
#### 6️⃣ EAR(눈 종횡비) 계산 함수 만들기 📐

이제 선택한 점들을 이용해서 **눈이 얼마나 떠졌는지(EAR)** 계산하는 함수를 만듭니다.
아래 공식을 코드로 옮겨보세요.
$$EAR = \frac{||p_2 - p_6|| + ||p_3 - p_5||}{2 \times ||p_1 - p_4||}$$

여기서 `p1`~`p6`는 여러분이 위에서 선택한 6개의 점을 순서대로 의미합니다.
* `p1`, `p4`: 눈의 양 끝점 (가로)
* `p2`, `p6`: 눈의 위아래 (세로 1)
* `p3`, `p5`: 눈의 위아래 (세로 2)

In [ ]:
def calculate_ear(pts):
    """
    눈 주변의 6개 좌표(pts)를 받아 EAR 값을 계산하여 반환하는 함수
    pts[0]: 왼쪽 끝 (p1)
    pts[1]: 위쪽 1 (p2)
    pts[2]: 위쪽 2 (p3)
    pts[3]: 오른쪽 끝 (p4)
    pts[4]: 아래쪽 2 (p5)
    pts[5]: 아래쪽 1 (p6)
    """
    
    # 1. 눈의 세로 길이 계산 (유클리드 거리)
    # dist.euclidean(점A, 점B) 함수 사용
    vertical_1 = dist.euclidean(pts[1], pts[5])
    vertical_2 = dist.euclidean(pts[2], pts[4])
    
    # 2. 눈의 가로 길이 계산
    horizontal = dist.euclidean(pts[0], pts[3])
    
    # 🎯 [2] EAR 공식 완성하기 ---
    # TODO: 위의 변수들을 사용하여 EAR 공식을 작성하세요.
    ear = _______
    
    return ear

---
#### 7️⃣ 결과 확인 🧪

이제 `open.png`와 `close.png`를 넣었을 때, 우리가 만든 모델과 알고리즘이 졸음을 잘 구분하는지 확인해 봅시다.

In [ ]:
def test_drowsiness_visualized(image_path, model):
    results = model(image_path)
    # 키포인트 좌표 가져오기
    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    
    # EAR 계산
    right_ear = calculate_ear(keypoints, RIGHT_EYE_IDXS)
    left_ear = calculate_ear(keypoints, LEFT_EYE_IDXS)
    
    # 양쪽 눈의 평균 사용
    avg_ear = (right_ear + left_ear) / 2.0
    
    # 이미지 준비
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.title(f"Result EAR: {avg_ear:.3f}", fontsize=15)

    for eye_idxs in [RIGHT_EYE_IDXS, LEFT_EYE_IDXS]:
        # 현재 눈의 6개 점 좌표만 가져옴 (순서대로 0~5번 인덱스가 됨)
        # 0:왼쪽끝, 1:위1, 2:위2, 3:오른쪽끝, 4:아래2, 5:아래1
        eye_points = keypoints[eye_idxs]
        
        # 1. 점 찍기 (빨간색)
        # eye_points[:, 0]은 x좌표들, eye_points[:, 1]은 y좌표들
        plt.scatter(eye_points[:, 0], eye_points[:, 1], c='red', s=30, zorder=3)
        
        # 2. 선 그리기 (노란색, EAR 공식에 쓰이는 선들)
        # 세로 선 1 (p2 <-> p6) : 인덱스 1번과 5번 연결
        plt.plot([eye_points[1,0], eye_points[5,0]], 
                 [eye_points[1,1], eye_points[5,1]], 
                 color='yellow', linewidth=2)
        
        # 세로 선 2 (p3 <-> p5) : 인덱스 2번과 4번 연결
        plt.plot([eye_points[2,0], eye_points[4,0]], 
                 [eye_points[2,1], eye_points[4,1]], 
                 color='yellow', linewidth=2)
                 
        # 가로 선 (p1 <-> p4) : 인덱스 0번과 3번 연결
        plt.plot([eye_points[0,0], eye_points[3,0]], 
                 [eye_points[0,1], eye_points[3,1]], 
                 color='yellow', linewidth=2)

    plt.axis('off')
    plt.show()
    
    return avg_ear

# 테스트 실행
print("=== 뜬 눈 테스트 (시각화 업그레이드) ===")
ear_open = test_drowsiness_visualized('../data/demo/iu_open.png', model)

print("\n=== 감은 눈 테스트 ===")
ear_close = test_drowsiness_visualized('../data/demo/iu_close.png', model)

print(f"\n[결과 분석] 뜬 눈({ear_open:.3f}) vs 감은 눈({ear_close:.3f})")

---
#### 8️⃣ 졸음 판단 영상 만들기

In [ ]:
def create_drowsiness_video(results, output_path, fps=30):
    
    # 영상 저장을 위한 설정 (건드리지 마세요!)
    if not results:
        print("❌ 결과 데이터가 없습니다.")
        return
    h, w = results[0].orig_img.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    
    # 🎯 [미션] 나만의 졸음 기준값 설정 ---
    # EAR 값이 이 숫자보다 작아지면 '졸음'으로 판단합니다. (예: 0.2 ~ 0.3)
    EAR_THRESH = _______

    print(f"🎬 영상 제작 시작... (기준값: {EAR_THRESH})")

    # 프레임별로 반복
    for result in results:
        frame = result.orig_img.copy() # 원본 이미지 가져오기
        
        # 사람이 감지되었을 때만 실행
        if result.keypoints is not None and len(result.keypoints.xy) > 0:
            # 1. 좌표 데이터 가져오기 (GPU -> CPU 변환)
            kpts = result.keypoints.xy[0].cpu().numpy()

            # 2. 양쪽 눈의 EAR 계산 (위에서 만든 함수 사용)
            right_ear = calculate_ear(kpts[RIGHT_EYE_IDXS])
            left_ear = calculate_ear(kpts[LEFT_EYE_IDXS])
            
            # 양쪽 눈의 평균 EAR 구하기
            avg_ear = (right_ear + left_ear) / 2.0

            # 🎯 [미션] 졸음 판단 로직 작성 ---
            # TODO: 평균 EAR(avg_ear)이 기준값(EAR_THRESH)보다 작으면 '경고', 아니면 '정상' 처리
            if avg_ear < _______: 
                color = (0, 0, 255) # 빨간색 (B, G, R 순서)
                message = "WAKE UP!"
                
                # 화면 중앙에 경고 메시지 띄우기
                cv2.putText(frame, message, (50, h//2), 
                            cv2.FONT_HERSHEY_SIMPLEX, 2, color, 4)
            else:
                color = (0, 255, 0) # 초록색
            
            # 3. 정보 시각화 (EAR 값 표시)
            cv2.putText(frame, f"EAR: {avg_ear:.2f}", (30, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

            # 4. 눈 위치에 점 찍기
            for idx in RIGHT_EYE_IDXS + LEFT_EYE_IDXS:
                x, y = int(kpts[idx][0]), int(kpts[idx][1])
                if x > 0 and y > 0:
                    cv2.circle(frame, (x, y), 3, color, -1)

        # 프레임 저장
        out.write(frame)

    out.release()
    print(f"✅ 저장 완료! 파일명: {output_path}")


results = model.predict('../data/demo/demo_hani.mp4',save=True)
create_drowsiness_video(results, 'result_custom.mp4', fps=30)